## Influence-driven data curation
Idea: the kernel of the surrogate model represents the relationship between outcomes across the factor space -> it helps us understand how the mean outcome across the factor space could change if one or more outcomes were improved (i.e. collected demos for)

Terms:
- $y_{target}$: the maximum outcome achievable
- $\mu(x)$: expected outcome at factor configuration $x$

For a single candidate demo $x^*$, the expected new mean outcome at any other point $x$ in the space is:
$$\mu_{new}(x) = \mu_{current}(x) + \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} (y_{target} - \mu_{current}(x^*))$$

To calculate the expected change in mean outcome across the entire factor space, we integrate that change over the whole domain $\mathcal{X}$.
$$\text{Total Influence}(x^*) = \left( y_{target} - \mu_{current}(x^*) \right) \cdot \int_{\mathcal{X}} \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} dx$$

To find the $x^*$ that maximizes the overall average change, simply calculate this score for all candidates and select the highest one.

What if you wanted to select more than one point (a batch $X_{batch} = [x_1, x_2, \dots, x_N]$) to collect demos for?
To score a full batch, you upgrade the scalar covariance $k(x, x^*)$ into a covariance matrix $K(X_{batch}, X_{batch})$. The expected total influence for a batch becomes:$$\text{Total Influence}(X_{batch}) = \int_{\mathcal{X}} K(x, X_{batch}) \left[ K(X_{batch}, X_{batch}) + \Sigma_{noise} \right]^{-1} (Y_{target} - M_{current}(X_{batch})) dx$$

In [11]:
import numpy as np
import torch
import pickle
import pandas as pd
from factors_config import FACTOR_COLUMNS, get_design_points_robot, BOUNDS, tkwargs

In [12]:
TASKS = ['pickblueblock', 'uprightcup', 'putgreeninpot']
MAX_OUTCOME = {'pickblueblock': 2.0, 'uprightcup': 3.0, 'putgreeninpot': 6.0}

In [31]:
# Load active model (currently only works for SingleTaskGP and FullyBayesianSingleTaskGP)
active_model_path = './results/putgreeninpot_active_offline_FullyBayesianSingleTaskGP_qBALD/run_1/models/trial_100_model.pkl'

with open(active_model_path, 'rb') as f:
    active_model_data = pickle.load(f)
    active_model = active_model_data['model']
    active_model_name = active_model_data.get('model_name', 'Unknown')
print(f"Loaded active model: {active_model_name}")

Loaded active model: FullyBayesianSingleTaskGP


In [32]:
# See covariance matrix on training data
bf_data = pd.read_csv('./results/putgreeninpot_bruteforce/results.csv')
print("Full factor space size: ", len(bf_data))
X_all = torch.tensor(bf_data[FACTOR_COLUMNS].values, **tkwargs)
y_all = torch.tensor(bf_data['continuous_outcome'].values, **tkwargs).unsqueeze(-1)

# Optional: Filter factor values i.e. restrict which rows of X_all greedy/local search may pick for the batch.
# Total influence is still summed over the full grid X_all (see calculate_batch_total_influence).
# Examples:
#   candidate_indices = np.where(bf_data['table_height'].values == 1)[0].tolist()
#   bottom_left = (bf_data['x'].values <= 0.5) & (bf_data['y'].values <= 0.5)
#   candidate_indices = np.where(bottom_left)[0].tolist()

# candidate_indices = None
candidate_indices = np.where(bf_data['camera_azimuth'].values == 90)[0].tolist()
print("Filtered factor space size: ", len(candidate_indices))

# active_model.covar_module(X_all).to_dense()

Full factor space size:  723
Filtered factor space size:  241


### Calculating batch total influence

In [21]:
def calculate_batch_total_influence(X_batch, active_model, X_all, y_target):
    """
    Scores a proposed batch of points based on expected total influence.

    X_all: full discrete grid over which the influence contribution is summed
    (the domain in the integral / sum in the markdown above). X_batch may be any subset
    of rows from this grid or elsewhere in factor space; batch entries must be valid inputs
    to the GP kernel.

    SingleTaskGP: standard (B, B) kernel and scalar / length-1 noise.
    FullyBayesianSingleTaskGP (and similar): kernel and noise are batched over MCMC
    samples (S, B, B) and (S, 1); we average the total-influence scalar over samples.
    """
    active_model.eval()
    active_model.likelihood.eval()

    with torch.no_grad():
        mean_batch = active_model.posterior(X_batch).mean
        K_bb = active_model.covar_module(X_batch).to_dense()
        K_ab = active_model.covar_module(X_all, X_batch).to_dense()
        noise_t = active_model.likelihood.noise

        if K_bb.ndim == 2:
            # (B, B), noise typically shape (1,) for homoskedastic SingleTaskGP
            noise = noise_t.to(dtype=K_bb.dtype, device=K_bb.device).mean().item()
            B = K_bb.shape[0]
            eye_B = torch.eye(B, device=K_bb.device, dtype=K_bb.dtype)
            K_bb_noisy = K_bb + noise * eye_B
            y_diff = y_target - mean_batch
            inv_term = torch.linalg.solve(K_bb_noisy, y_diff)
            influence_per_point = torch.matmul(K_ab, inv_term)
            total_influence = influence_per_point.sum()
        else:
            # (S, B, B) e.g. FullyBayesianSingleTaskGP with S MCMC hyperparameter samples
            S, B, _ = K_bb.shape
            noise = noise_t.squeeze(-1).to(dtype=K_bb.dtype, device=K_bb.device)
            eye_B = torch.eye(B, device=K_bb.device, dtype=K_bb.dtype)
            K_bb_noisy = K_bb + noise.view(S, 1, 1) * eye_B
            y_diff = y_target - mean_batch
            inv_term = torch.linalg.solve(K_bb_noisy, y_diff)
            influence_per_point = torch.matmul(K_ab, inv_term)
            total_influence = influence_per_point.sum(dim=1).mean()

    return total_influence.item()

### Greedy and stochastic greedy batch search

**Integration vs candidate pool:** `candidate_indices` (optional filtered set of `X_all`, restricting which rows of `X_all` greedy search may add to the batch). `calculate_batch_total_influence` always sums influence over the **full** `X_all`.

In [17]:
# Greedy batch selection
def propose_batch_greedy(batch_size, active_model, X_all, y_target, candidate_indices=None):
    """
    Greedily builds a batch of size `batch_size` that maximizes total influence.

    X_all: full discrete grid; total influence is summed over all rows (same tensor passed to
    calculate_batch_total_influence as X_all).
    candidate_indices: optional list of row indices into X_all that may be added to the batch.
        If None, every row of X_all is eligible.
    """
    n = X_all.shape[0]
    if candidate_indices is None:
        pool = list(range(n))
    else:
        pool = list(dict.fromkeys(int(i) for i in candidate_indices))
    for idx in pool:
        if not (0 <= idx < n):
            raise ValueError(f"candidate index {idx} out of range for X_all with {n} rows.")
    if batch_size > len(pool):
        raise ValueError(
            f"batch_size ({batch_size}) cannot exceed number of candidate indices ({len(pool)})."
        )

    selected_indices = []
    selected_set = set()

    print(f"Proposing a batch of {batch_size} points (candidate pool size {len(pool)})...")

    for step in range(batch_size):
        best_score = -float('inf')
        best_idx = -1

        for i in pool:
            if i in selected_set:
                continue

            current_batch_indices = selected_indices + [i]
            X_batch_candidate = X_all[current_batch_indices]

            score = calculate_batch_total_influence(X_batch_candidate, active_model, X_all, y_target)

            if score > best_score:
                best_score = score
                best_idx = i

        selected_indices.append(best_idx)
        selected_set.add(best_idx)
        print(f"  Step {step+1}/{batch_size}: Added point index {best_idx} | New Batch Influence Score: {best_score:.4f}")

    X_proposed = X_all[selected_indices]
    return X_proposed, selected_indices, best_score

In [18]:
# Stochastic greedy batch selection
import random

def propose_batch_local_search(batch_size, active_model, X_all, y_target, max_iters=1000, candidate_indices=None):
    """
    Improves a batch using Stochastic Local Search (Swapping) over a discrete grid.

    candidate_indices: same as propose_batch_greedy; swaps only replace with other candidates.
    """
    n = X_all.shape[0]
    if candidate_indices is None:
        pool = list(range(n))
    else:
        pool = list(dict.fromkeys(int(i) for i in candidate_indices))
    all_indices = set(pool)

    # 1. Initialize with the Greedy approach for a strong head start
    print("Initializing with Greedy search...")
    _, current_indices, best_score = propose_batch_greedy(
        batch_size, active_model, X_all, y_target, candidate_indices=pool
    )

    print(f"\nStarting Local Search Optimization. Initial Score: {best_score:.4f}")

    # 2. Iteratively attempt to swap points to find synergistic combinations
    improvements = 0

    for iteration in range(max_iters):
        idx_to_remove = random.choice(current_indices)

        available_indices = list(all_indices - set(current_indices))
        if not available_indices:
            break
        idx_to_add = random.choice(available_indices)

        proposed_indices = current_indices.copy()
        proposed_indices.remove(idx_to_remove)
        proposed_indices.append(idx_to_add)

        proposed_X_batch = X_all[proposed_indices]
        proposed_score = calculate_batch_total_influence(proposed_X_batch, active_model, X_all, y_target)

        if proposed_score > best_score:
            current_indices = proposed_indices
            best_score = proposed_score
            improvements += 1
            print(f"  [Iter {iteration}] Improved! Swapped {idx_to_remove} for {idx_to_add}. New Score: {best_score:.4f}")

    print(f"\nLocal Search Complete. Made {improvements} improvements.")

    final_X_batch = X_all[current_indices]
    return final_X_batch, current_indices, best_score

### Test these selection strategies out

In [19]:
# Define your target/max outcome (e.g. 3.0 for the 'uprightcup' task)
Y_TARGET = 6.0  # change for different tasks
BATCH_SIZE = 23  # how many points to collect demos at

In [33]:
# --- Greedy selection ---
greedy_X, greedy_indices, greedy_score = propose_batch_greedy(
    batch_size=BATCH_SIZE,
    active_model=active_model,
    X_all=X_all,
    y_target=Y_TARGET,
    candidate_indices=candidate_indices,
)

print(f"Greedy Score: {greedy_score:.4f}")
# Show proposed points in a table
# print("\nProposed Factor Configurations to Collect Next:")
# print(pd.DataFrame(data=proposed_X.numpy(), columns=FACTOR_COLUMNS))

Proposing a batch of 23 points (candidate pool size 241)...
  Step 1/23: Added point index 591 | New Batch Influence Score: 264.9571
  Step 2/23: Added point index 549 | New Batch Influence Score: 392.7501
  Step 3/23: Added point index 699 | New Batch Influence Score: 502.0830
  Step 4/23: Added point index 492 | New Batch Influence Score: 537.5987
  Step 5/23: Added point index 649 | New Batch Influence Score: 564.7902
  Step 6/23: Added point index 634 | New Batch Influence Score: 586.9874
  Step 7/23: Added point index 554 | New Batch Influence Score: 598.8704
  Step 8/23: Added point index 569 | New Batch Influence Score: 609.7071
  Step 9/23: Added point index 722 | New Batch Influence Score: 620.2186
  Step 10/23: Added point index 491 | New Batch Influence Score: 626.9391
  Step 11/23: Added point index 660 | New Batch Influence Score: 632.3890
  Step 12/23: Added point index 558 | New Batch Influence Score: 636.8106
  Step 13/23: Added point index 638 | New Batch Influence Sco

In [23]:
# --- Stochastic greedy selection ---
stoch_greedy_X, stoch_greedy_indices, stoch_greedy_score = propose_batch_local_search(
    batch_size=BATCH_SIZE,
    active_model=active_model,
    X_all=X_all,
    y_target=Y_TARGET,
    max_iters=1000,  # Adjust based on your compute budget
    candidate_indices=candidate_indices,
)

print(f"Stochastic Greedy Score: {stoch_greedy_score:.4f}")
# print("\nProposed Factor Configurations to Collect Next:")
# print(pd.DataFrame(data=optimized_X.numpy(), columns=FACTOR_COLUMNS))

Initializing with Greedy search...
Proposing a batch of 23 points (candidate pool size 241)...
  Step 1/23: Added point index 591 | New Batch Influence Score: 264.9571
  Step 2/23: Added point index 549 | New Batch Influence Score: 392.7501
  Step 3/23: Added point index 699 | New Batch Influence Score: 502.0830
  Step 4/23: Added point index 492 | New Batch Influence Score: 537.5987
  Step 5/23: Added point index 649 | New Batch Influence Score: 564.7902
  Step 6/23: Added point index 634 | New Batch Influence Score: 586.9874
  Step 7/23: Added point index 554 | New Batch Influence Score: 598.8704
  Step 8/23: Added point index 569 | New Batch Influence Score: 609.7071
  Step 9/23: Added point index 722 | New Batch Influence Score: 620.2186
  Step 10/23: Added point index 491 | New Batch Influence Score: 626.9391
  Step 11/23: Added point index 660 | New Batch Influence Score: 632.3890
  Step 12/23: Added point index 558 | New Batch Influence Score: 636.8106
  Step 13/23: Added point 

In [24]:
# Compare to "certain failures" selection
certain_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_certain_failures.csv')
print(f"Loaded {len(certain_failures_points)} points")
cf_X = torch.tensor(certain_failures_points[FACTOR_COLUMNS].values, **tkwargs)
cf_score = calculate_batch_total_influence(cf_X, active_model, X_all, Y_TARGET)
print(f"Certain Failures Score: {cf_score:.4f}")

Loaded 23 points
Certain Failures Score: 454.2716


In [25]:
# Compare to "observed failures" selection
observed_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_observed_failures.csv')
print(f"Loaded {len(observed_failures_points)} points")
of_X = torch.tensor(observed_failures_points[FACTOR_COLUMNS].values, **tkwargs)
of_score = calculate_batch_total_influence(of_X, active_model, X_all, Y_TARGET)
print(f"Observed Failures Score: {of_score:.4f}")

Loaded 23 points
Observed Failures Score: 548.2801


In [26]:
# Compare to random selection (same candidate pool as greedy when candidate_indices is set)
random_scores = []
if candidate_indices is None:
    pool_idx = torch.arange(len(X_all))
else:
    pool_idx = torch.tensor(candidate_indices, dtype=torch.long)
if BATCH_SIZE > len(pool_idx):
    raise ValueError("BATCH_SIZE cannot exceed len(candidate pool).")

for i in range(10):
    perm = torch.randperm(len(pool_idx))[:BATCH_SIZE]
    random_indices = pool_idx[perm]
    random_X_batch = X_all[random_indices]
    random_score = calculate_batch_total_influence(random_X_batch, active_model, X_all, Y_TARGET)
    random_scores.append(random_score)

random_score = sum(random_scores) / len(random_scores)
print(f"Random Selection Score: {random_score:.4f}")

Random Selection Score: 540.4041


In [27]:
# Compare all influence scores
print("--- Influence Scores ---")
print(f"Greedy: {greedy_score:.4f}")
print(f"Stochastic Greedy: {stoch_greedy_score:.4f}")
print(f"Certain Failures: {cf_score:.4f}")
print(f"Observed Failures: {of_score:.4f}")
print(f"Random Selection: {random_score:.4f}")

--- Influence Scores ---
Greedy: 660.2835
Stochastic Greedy: 660.2835
Certain Failures: 454.2716
Observed Failures: 548.2801
Random Selection: 540.4041
